# Design Summary
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

The final rollup of the design pipeline. Everything printed here is
**read from the `out/` handoffs the preceding fourteen notebooks wrote**
(plus one sizing-loop re-run to echo the closure, same pattern as every
notebook since NB2) — this notebook adds no new physics and writes no
handoff. It exists so a reader (or a release snapshot) has one place with:

1. the converged design point and the as-selected reality next to it,
2. the control / vibration / thermal / drag margins in one table,
3. the frozen COTS hardware list, and
4. **every standing finding** the pipeline carries, collected
   programmatically from the handoffs — nothing here is hand-maintained.

## Inputs

Everything in `out/` (NB1–NB14). No outputs.

---

In [1]:
import sys, math
from pathlib import Path
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
import yaml
import pandas as pd

REPO_ROOT   = Path().resolve().parents[0]
SRC_PATH    = REPO_ROOT / "src"
CONFIG_PATH = REPO_ROOT / "config"
OUT_PATH    = REPO_ROOT / "out"
sys.path.insert(0, str(SRC_PATH))

plt.rcParams.update({
    "figure.dpi"        : 120,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.alpha"        : 0.25,
    "font.size"         : 10,
})
C = ["#2c7bb6", "#d7191c", "#1a9641", "#f68b33", "#762a83"]

FIG_DIR = Path("figures")   # per-notebook figures directory
FIG_DIR.mkdir(exist_ok=True)


# Section 1 — Design Point at a Glance

Re-run the sizing loop from `config/` and cross-check it against the
frozen-hardware echo (`out/components.yaml`) — a mismatch means stale
handoffs, not new physics.

---

In [2]:
from conceptual_design import (
    run_sizing_loop,
    Environment, Mission, Aerodynamics, Battery,
    WeightFraction, PropulsiveSystemParameters,
)
from conceptual_design.forward_flight_power import ForwardFlightParams
from conceptual_design.wing_sizing import WingStructureParams
from conceptual_design.models import RotorParams, Avionics

env     = Environment()
mission = Mission.from_yaml(CONFIG_PATH / "mission.yaml")
aero    = Aerodynamics.from_yaml(CONFIG_PATH / "aerodynamics.yaml")
batt    = Battery.from_yaml(CONFIG_PATH / "battery.yaml")
wf      = WeightFraction.from_yaml(CONFIG_PATH / "initial_weight_fraction_estimation.yaml")
prop    = PropulsiveSystemParameters.from_yaml(CONFIG_PATH / "propulsive_system_parameters.yaml")
ff      = ForwardFlightParams.from_yaml(CONFIG_PATH / "forward_flight_params.yaml")
ws      = WingStructureParams.from_yaml(CONFIG_PATH / "wing_structure_params.yaml")
rotor    = RotorParams.from_yaml(CONFIG_PATH / "rotor.yaml")
avionics = Avionics.from_yaml(CONFIG_PATH / "avionics.yaml")

result = run_sizing_loop(
    m_payload_kg=mission.payload_kg, mission=mission, aero=aero, batt=batt,
    wf=wf, prop_params=prop, ff_params=ff, ws_params=ws, env=env,
    D_rotor_m=rotor.D_rotor_m, P_hotel_W=avionics.P_hotel_W,
)

def load(name):
    return yaml.safe_load((OUT_PATH / name).read_text(encoding="utf-8"))

af       = load("airfoil.yaml")
vanes    = load("control_vanes.yaml")
ail      = load("aileron.yaml")
ail_cots = load("aileron_cots.yaml")
vib      = load("vibration.yaml")
vib_cots = load("vibration_cots.yaml")
fus      = load("fuselage.yaml")
fus_cots = load("fuselage_cots.yaml")
thermal  = load("thermal.yaml")
elec     = load("electrical.yaml")
massprop = load("mass_properties.yaml")
comp     = load("components.yaml")

# handoff freshness: the COTS freeze must echo this run's closure
assert abs(comp["design_point"]["MTOW_kg"] - result.m_total_kg) < 5e-3, (
    "out/components.yaml design-point echo != sizing re-run -- stale handoffs")

as_sel = fus_cots["as_selected"]
t_mission = mission.t_hover + mission.t_transition + mission.t_cruise

print("VEHICLE AT A GLANCE")
print("-" * 62)
print(f"{'MTOW (mass closure)':<34}: {result.m_total_kg:.3f} kg")
print(f"{'All-up, as-selected hardware':<34}: {as_sel['m_items_total_kg']:.3f} kg "
      f"({as_sel['m_delta_kg']*1e3:+.0f} g standing findings)")
print(f"{'Hover / design power':<34}: {result.P_hover_W:.0f} / {result.P_design_W:.0f} W")
print(f"{'Pack':<34}: {elec['battery_series_cells']}S {elec['pack_voltage_v']:.1f} V, "
      f"{elec['pack_capacity_ah']:.2f} Ah sized (hover {elec['hover_current_a']:.1f} A)")
print(f"{'Mission':<34}: {mission.t_hover:.0f} s hover + {mission.t_transition:.0f} s "
      f"transition + {mission.t_cruise:.0f} s cruise = {t_mission/60:.1f} min")
print(f"{'Wing':<34}: b {result.wing.b_wing*1e3:.0f} mm, S {result.wing.S_wing:.4f} m^2, "
      f"MAC {result.wing.chord_mean*1e3:.1f} mm, {af['designation']}")
print(f"{'Rotor':<34}: {rotor.D_rotor_m*1e3:.0f} mm, {comp['selected']['propeller']['name']}")
print(f"{'Fuselage (conceptual / as-sel.)':<34}: {fus['D_fus_m']*1e3:.0f} x {fus['L_fus_m']*1e3:.0f} mm  /  "
      f"{fus_cots['D_fus_m']*1e3:.0f} x {fus_cots['L_fus_m']*1e3:.0f} mm")
print(f"{'CG / static margin':<34}: {fus['x_CG_m']*1e3:.1f} mm from nose, "
      f"{fus['static_margin']*100:.0f}% MAC")
print(f"{'Inertia (body FRD, about CG)':<34}: Ixx {massprop['inertia_about_CG_body_FRD']['Ixx_kgm2']:.4f}, "
      f"Iyy {massprop['inertia_about_CG_body_FRD']['Iyy_kgm2']:.4f}, "
      f"Izz {massprop['inertia_about_CG_body_FRD']['Izz_kgm2']:.4f} kg m^2")
print(f"{'Construction':<34}: {wf.construction_method} (ADR-0008/0010), "
      f"structure {fus['m_shell_kg']*1e3:.0f} g (as-sel. hull {fus_cots['m_shell_kg']*1e3:.0f} g)")


VEHICLE AT A GLANCE
--------------------------------------------------------------
MTOW (mass closure)               : 2.303 kg
All-up, as-selected hardware      : 2.427 kg (+125 g standing findings)
Hover / design power              : 652 / 652 W
Pack                              : 6S 22.2 V, 3.50 Ah sized (hover 29.4 A)
Mission                           : 120 s hover + 40 s transition + 900 s cruise = 17.7 min
Wing                              : b 1049 mm, S 0.1834 m^2, MAC 174.8 mm, NACA 2412
Rotor                             : 203 mm, Master Airscrew 3-blade 8x6 prop (in airframe duct, EST)
Fuselage (conceptual / as-sel.)   : 96 x 478 mm  /  106 x 531 mm
CG / static margin                : 236.6 mm from nose, 5% MAC
Inertia (body FRD, about CG)      : Ixx 0.0214, Iyy 0.0402, Izz 0.0517 kg m^2
Construction                      : segmented_fdm (ADR-0008/0010), structure 317 g (as-sel. hull 375 g)


# Section 2 — Mass: Closure Allocations vs. As-Selected Hardware

Same idiom as the NB11 budget chart (bar = actual hardware, tick =
allocation, red = over), for the five COTS groups of the freeze. The +/-
labels carry the numbers so the verdict never rides on color alone; the
all-up rollup (which would dwarf the group bars) is reported as text.

---

In [3]:
budgets = comp["budgets"]
groups  = ["avionics_bay", "esc", "motor_fan", "servo_each", "battery"]
labels  = {
    "avionics_bay": "avionics bay",
    "esc":          "ESC",
    "motor_fan":    "EDF motor + prop",
    "servo_each":   "servo (each)",
    "battery":      "battery pack",
}

fig, ax = plt.subplots(figsize=(7.5, 3.6))
y = np.arange(len(groups))[::-1]
actual = [budgets[g]["actual_g"] for g in groups]
alloc  = [budgets[g]["alloc_g"]  for g in groups]
colors = [C[0] if budgets[g]["within"] else C[1] for g in groups]

ax.barh(y, actual, height=0.55, color=colors)
ax.plot(alloc, y, "k|", markersize=22, markeredgewidth=2.2, ls="none", zorder=3)
for yi, g in zip(y, groups):
    ax.annotate(f"{budgets[g]['margin_g']:+.0f} g",
                (max(budgets[g]['actual_g'], budgets[g]['alloc_g']), yi),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=9)
ax.set_yticks(y, [labels[g] for g in groups])
ax.set_xlabel("mass [g]")
ax.set_title("As-selected hardware vs mass-closure allocations\n"
             "(bar = actual, tick = allocation; red = over)")
fig.tight_layout()
fig.savefig(FIG_DIR / "design_summary_mass.png", bbox_inches="tight")
plt.show()

for g in groups:
    b = budgets[g]
    print(f"{'ok      ' if b['within'] else 'FINDING '}{g:14s}: "
          f"{b['actual_g']:7.1f} g vs {b['alloc_g']:7.1f} g ({b['margin_g']:+7.1f} g)")
print(f"\nAll-up rollup (NB14): {as_sel['m_items_total_kg']*1e3:.0f} g as-selected vs "
      f"{as_sel['m_closure_mtow_kg']*1e3:.0f} g closure MTOW "
      f"({as_sel['m_delta_kg']*1e3:+.0f} g)")


FINDING avionics_bay  :   138.6 g vs    96.2 g (  -42.4 g)
ok      esc           :    30.0 g vs    86.3 g (  +56.3 g)
FINDING motor_fan     :   306.0 g vs   207.2 g (  -98.8 g)
ok      servo_each    :     8.9 g vs    12.0 g (   +3.1 g)
FINDING battery       :   613.0 g vs   554.6 g (  -58.4 g)

All-up rollup (NB14): 2427 g as-selected vs 2303 g closure MTOW (+125 g)


D:\Temp\ipykernel_7872\206269572.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Section 3 — Margins in One Table

Control authority (hover and cruise), servo torque, vibration isolation,
thermal paths, and the cruise drag budget — each against the requirement
its own notebook checked, with the notebook that owns it.

---

In [4]:
CD0_total = af["Cd0_section"] * 1.10 + fus_cots["CD0_fus"] + 0.0025   # NB14 buildup (as-selected hull)
rows = [
    ("hover roll authority",      "NB3",  f"{vanes['ddot_roll_deg_s2']:.0f} deg/s^2",
     f">= {ail['ddot_min_deg_s2']:.0f}", vanes["ddot_roll_deg_s2"] >= ail["ddot_min_deg_s2"]),
    ("hover pitch authority",     "NB3",  f"{vanes['ddot_pitch_deg_s2']:.0f} deg/s^2",
     f">= {ail['ddot_min_deg_s2']:.0f}", vanes["ddot_pitch_deg_s2"] >= ail["ddot_min_deg_s2"]),
    ("cruise roll (ail + vane)",  "NB12", f"{ail_cots['ddot_roll_total_cruise_deg_s2']:.0f} deg/s^2",
     f">= {ail_cots['ddot_min_deg_s2']:.0f}", ail_cots["cruise_authority_ok"]),
    ("servo stall vs vane req",   "NB12", f"x{ail_cots['cots_servo']['margin_vs_vane_req']:.1f}",
     ">= x1.0", ail_cots["cots_servo"]["margin_vs_vane_req"] >= 1.0),
    ("1/rev isolation (FC/IMU)",  "NB13", f"{vib_cots['modules']['fc_imu']['attenuation_pct']:.0f}% att.",
     f">= {100*(1-vib_cots['target_transmissibility']):.0f}%", vib_cots["window_ok"]),
    ("battery bay venting",       "NB7",  f"{thermal['battery']['temp_margin_C']:.1f} C margin",
     "> 0 C", thermal["battery"]["ok"]),
    ("ESC cold-plate",            "NB7",  f"{thermal['esc']['temp_margin_C']:.1f} C margin",
     "mass in alloc", thermal["esc"]["ok"]),
    ("cruise drag budget",        "NB14", f"CD0 {CD0_total:.4f}",
     f"<= {aero.CD0:.4f}", CD0_total <= aero.CD0),
    ("frozen parts fit the hull", "NB14", f"{sum(1 for e in fus_cots['bay_fit'] if e['ok'])}/"
     f"{len(fus_cots['bay_fit'])} fit", "all fit", all(e["ok"] for e in fus_cots["bay_fit"])),
    ("structure vs budget (as-sel.)", "NB14",
     f"{fus_cots['m_shell_kg']*1e3:.0f} g",
     f"<= {(fus_cots['m_struct_pool_kg'] - fus_cots['m_struct_carved_kg'])*1e3:.0f} g",
     fus_cots["m_shell_kg"] <= fus_cots["m_struct_pool_kg"] - fus_cots["m_struct_carved_kg"]),
]
print(f"{'check':<28}{'owner':<7}{'value':<22}{'requirement':<16}{'verdict'}")
print("-" * 82)
for name, owner, val, req, ok in rows:
    print(f"{name:<28}{owner:<7}{val:<22}{req:<16}{'ok' if ok else 'FINDING'}")


check                       owner  value                 requirement     verdict
----------------------------------------------------------------------------------
hover roll authority        NB3    868 deg/s^2           >= 30           ok
hover pitch authority       NB3    615 deg/s^2           >= 30           ok
cruise roll (ail + vane)    NB12   560 deg/s^2           >= 30           ok
servo stall vs vane req     NB12   x3.2                  >= x1.0         ok
1/rev isolation (FC/IMU)    NB13   90% att.              >= 90%          ok
battery bay venting         NB7    19.5 C margin         > 0 C           ok
ESC cold-plate              NB7    14.7 C margin         mass in alloc   FINDING
cruise drag budget          NB14   CD0 0.0226            <= 0.0250       ok
frozen parts fit the hull   NB14   4/4 fit               all fit         ok
structure vs budget (as-sel.)NB14   375 g                 <= 342 g        FINDING


# Section 4 — Frozen COTS Hardware

The procurement list as frozen by NB11 (`out/components.yaml`); `frozen`
means pinned via `selection.frozen` rather than auto-selected. After
procurement: weigh the parts, fix the `EST` entries, pin the remaining
ids, update the regression pins — the diff is the procurement record.

---

In [5]:
rows = []
for cat, s in comp["selected"].items():
    rows.append({
        "category": cat, "part": s["name"], "id": s["id"],
        "mass_g": s["mass_g"], "pinned": s["frozen"],
        "alternatives": len(s["feasible_alternatives"]),
    })
rows.append({"category": "supporting avionics", "part": "GPS/telemetry/RX/airspeed/PM/looms",
             "id": "-", "mass_g": sum(e["mass_g"] for e in comp["supporting_avionics"]),
             "pinned": False, "alternatives": 0})
display(pd.DataFrame(rows).set_index("category"))


,part,id,mass_g,pinned,alternatives
category,,,,,
flight_controller,Holybro Pixhawk 6C (plastic case),pixhawk_6c,34.6,False,4
esc,APD 80F3[X] (KISS/serial telemetry),apd_80f3x,30.0,False,1
edf_motor,SunnySky X4120 KV465 (EST),sunnysky_x4120_465,283.0,False,3
propeller,Master Airscrew 3-blade 8x6 prop (in airframe ...,ma_3blade_8x6,23.0,True,1
servo,"KST X08 V6 (metal gear, digital)",kst_x08_v6,8.9,False,3
battery,Gens Ace 4000 mAh 6S 30C LiPo,gens_ace_6s_4000_30c,613.0,False,3
supporting avionics,GPS/telemetry/RX/airspeed/PM/looms,-,104.0,False,0


# Section 5 — Standing Findings

Collected **programmatically** from the handoffs — this list cannot go
stale. Each finding indicts an *assumption* (a weight fraction, a
packaging density, an allocation), not a part: the selections were
already the lightest physics allows.

---

In [6]:
findings = []

for g in ("avionics_bay", "esc", "motor_fan", "servo_each", "battery"):
    b = comp["budgets"][g]
    if not b["within"]:
        findings.append(f"{g}: {b['actual_g']:.0f} g vs {b['alloc_g']:.0f} g allocation "
                        f"({b['margin_g']:+.0f} g) -- {b.get('note', 'allocation finding')}")

if not thermal["esc"]["ok"]:
    findings.append(f"ESC cold-plate (ADR-0009): needs {thermal['esc']['A_req_cm2']:.0f} cm^2 / "
                    f"{thermal['esc']['plate_mass_g']:.0f} g plate vs the ESC allocation; "
                    f"temp margin only {thermal['esc']['temp_margin_C']:.1f} C")

for e in fus_cots["bay_fit"]:
    if not e["ok"]:
        findings.append(f"fit: {e['part']} does not fit {e['where']} "
                        f"(needs {e['need_mm']} mm, has {e['have_mm']} mm)")

if fus_cots["as_selected"]["m_delta_kg"] > 0:
    findings.append(f"as-selected all-up mass {fus_cots['as_selected']['m_items_total_kg']:.3f} kg "
                    f"is {fus_cots['as_selected']['m_delta_kg']*1e3:+.0f} g over the closure MTOW "
                    f"-- the sum of the mass findings above")

m_struct_budget_cots = fus_cots["m_struct_pool_kg"] - fus_cots["m_struct_carved_kg"]
if fus_cots["m_shell_kg"] > m_struct_budget_cots:
    findings.append(f"as-selected structure model {fus_cots['m_shell_kg']*1e3:.0f} g exceeds the "
                    f"non-wing structural budget {m_struct_budget_cots*1e3:.0f} g -- the hull that "
                    f"holds the real hardware outgrows the structural fraction at the closure MTOW")

d = fus_cots["delta_vs_conceptual"]
if abs(d["D_fus_mm"]) > 0.02 * fus["D_fus_m"] * 1e3:
    findings.append(f"as-selected hull grows to {fus_cots['D_fus_m']*1e3:.0f} x "
                    f"{fus_cots['L_fus_m']*1e3:.0f} mm ({d['D_fus_mm']:+.1f} / {d['L_fus_mm']:+.1f} mm "
                    f"vs conceptual) -- rigid COTS envelopes drive the bay stack; CAD/CFD stay on "
                    f"the conceptual geometry until a config revision adopts this (ADR-0012)")

print(f"{len(findings)} standing finding(s):\n")
for i, f_ in enumerate(findings, 1):
    print(f"{i:2d}. {f_}")


7 standing finding(s):

 1. avionics_bay: 139 g vs 96 g allocation (-42 g) -- FC + supporting avionics vs the net avionics-bay budget
 2. motor_fan: 306 g vs 207 g allocation (-99 g) -- EDF motor + fan/prop unit vs the motor_fan layout allocation
 3. battery: 613 g vs 555 g allocation (-58 g) -- COTS pack vs the mass-closure battery mass (capacity quantisation makes a small overshoot expected)
 4. ESC cold-plate (ADR-0009): needs 213 cm^2 / 115 g plate vs the ESC allocation; temp margin only 14.7 C
 5. as-selected all-up mass 2.427 kg is +125 g over the closure MTOW -- the sum of the mass findings above
 6. as-selected structure model 375 g exceeds the non-wing structural budget 342 g -- the hull that holds the real hardware outgrows the structural fraction at the closure MTOW
 7. as-selected hull grows to 106 x 531 mm (+10.6 / +53.1 mm vs conceptual) -- rigid COTS envelopes drive the bay stack; CAD/CFD stay on the conceptual geometry until a config revision adopts this (ADR-0012)


# Section 6 — Closing Summary

---

In [7]:
bar = "=" * 66
print(bar)
print("  V-BAT-LIKE TAIL-SITTER -- FROZEN CONCEPTUAL DESIGN POINT".center(66))
print(bar)
print(f"  {'MTOW (closure) / as-selected':<36}: {result.m_total_kg:.3f} / "
      f"{as_sel['m_items_total_kg']:.3f} kg")
print(f"  {'Hover power / pack':<36}: {result.P_hover_W:.0f} W / "
      f"{elec['battery_series_cells']}S {elec['pack_capacity_ah']:.2f} Ah")
print(f"  {'Wing / rotor':<36}: b {result.wing.b_wing*1e3:.0f} mm / "
      f"D {rotor.D_rotor_m*1e3:.0f} mm ({comp['selected']['propeller']['id']})")
print(f"  {'Fuselage (conceptual/as-sel.)':<36}: {fus['D_fus_m']*1e3:.0f}x{fus['L_fus_m']*1e3:.0f} / "
      f"{fus_cots['D_fus_m']*1e3:.0f}x{fus_cots['L_fus_m']*1e3:.0f} mm")
print(f"  {'Standing findings':<36}: {len(findings)}")
print(bar)
print("  Next iteration: procure & weigh the frozen parts, fix the EST")
print("  entries, then fold the as-selected deltas (battery envelope,")
print("  motor mass, avionics stack) into config/ as a reviewed change --")
print("  the pipeline re-converges from there (ADR-0012).")
print(bar)


      V-BAT-LIKE TAIL-SITTER -- FROZEN CONCEPTUAL DESIGN POINT    
  MTOW (closure) / as-selected        : 2.303 / 2.427 kg
  Hover power / pack                  : 652 W / 6S 3.50 Ah
  Wing / rotor                        : b 1049 mm / D 203 mm (ma_3blade_8x6)
  Fuselage (conceptual/as-sel.)       : 96x478 / 106x531 mm
  Standing findings                   : 7
  Next iteration: procure & weigh the frozen parts, fix the EST
  entries, then fold the as-selected deltas (battery envelope,
  motor mass, avionics stack) into config/ as a reviewed change --
  the pipeline re-converges from there (ADR-0012).
